In [1]:
import pandas as pd 
import numpy as np 
import os 

In [ ]:
chunks = [] 
for chunk in pd.read_csv('bill_schedule.csv',chunksize = 500_000):
    chunk['io_time'] = pd.to_datetime(chunk['io_time']) 
    chunks.append(chunk) 

sche = pd.concat(chunks, ignore_index=True) 
del chunks 
sche = sche.sort_values(['bill_code', 'io_time']).reset_index(drop=True) 


sche['pre_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(1) 
sche_unique = sche[sche['warehouse_name'] != sche['pre_wh']].copy() 
sche_unique['rank'] = sche_unique.groupby('bill_code').cumcount() 


In [46]:
max_rank = sche_unique.groupby('bill_code')['rank'].transform(max) 
sche_unique['rank_re'] = max_rank - sche_unique['rank'] 

C:\Users\VIET ANH\AppData\Local\Temp\ipykernel_26132\1415269964.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  max_rank = sche_unique.groupby('bill_code')['rank'].transform(max)


In [47]:
wh_1a = pd.read_csv('warehouse_1a.csv') 
set_1a = set(wh_1a['name']) 
sche_1a = sche_unique[sche_unique['warehouse_name'].isin(set_1a)].copy()
count_1a = sche_1a.groupby('bill_code').size().reset_index(name = 'count_1a') 
count_1a = count_1a[count_1a['count_1a'] >= 2] 
sche_unique = sche_unique[sche_unique['bill_code'].isin(set(count_1a['bill_code']))]


In [48]:
traces = pd.read_csv('output_all_traces/bill_trunk_traces.csv')

In [49]:
traces_1a = traces[traces['bill_code'].isin(set(sche_unique['bill_code']))] 
traces_1a = traces_1a[~traces_1a['first_trunk'].isin(set_1a)] 
traces_1a = traces_1a[~traces_1a['last_trunk'].isin(set_1a)] 
traces_1a.to_csv(os.path.join('output_all_traces', 'traces_1a.csv'), index = False)

In [50]:
origin = pd.read_csv("output_all_traces/origin_to_1A.csv") 
des = pd.read_csv('output_all_traces/1A_to_destination.csv') 

In [51]:

print(traces_1a.shape) 
print(origin.shape)
print(des.shape)
print(sche_unique['bill_code'].nunique())

(517792, 7)
(738014, 7)
(672255, 7)
1083170


In [52]:
sche_final = sche_unique[sche_unique['bill_code'].isin(set(traces_1a['bill_code']))].copy()

In [57]:
def make_data(sche, title): 
    sche_1a_filter = sche[sche['warehouse_name'].isin(set_1a)]
    first = sche_1a_filter.groupby('bill_code')['rank'].min()
    last = sche_1a_filter.groupby('bill_code')['rank_re'].min()
    rank_first = first.value_counts().sort_index().reset_index()
    rank_last = last.value_counts().sort_index().reset_index()
    rank_first.columns = ['rank_to_1A', 'so_luong_bill']
    rank_last.columns = ['rank_1A_to', 'so_luong_bill']
    print(title)
    print(rank_first)
    print(rank_last)
    print()
    return rank_first, rank_last


through_least_2_first, through_least_2_last = make_data(sche_unique, "through at least two 1A")
traces_least_2_first, traces_least_2_last = make_data(sche_final, "traces well") 


through at least two 1A
   rank_to_1A  so_luong_bill
0           0         319459
1           1         738014
2           2          25546
3           3            140
4           4              7
5           5              2
6           6              2
    rank_1A_to  so_luong_bill
0            0         351019
1            1         672255
2            2          56702
3            3           2371
4            4            643
5            5            117
6            6             43
7            7             11
8            8              6
9            9              1
10          12              2

traces well
   rank_to_1A  so_luong_bill
0           1         501006
1           2          16690
2           3             86
3           4              6
4           5              2
5           6              2
   rank_1A_to  so_luong_bill
0           1         475372
1           2          40197
2           3           1672
3           4            429
4           5          

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6)) 
sns.barplot(data = traces_least_2_first, x="rank_to_1A", y = 'so_luong_bill')
plt.title('Thống kê số lượng Bill theo Rank của Kho 1A đầu tiên (origin -> 1A)', fontsize=14, fontweight='bold')
plt.xlabel('Rank của Kho 1A đầu tiên', fontsize=12)
plt.ylabel('Số lượng Bill', fontsize=12)
for index, row in traces_least_2_first.iterrows(): 
    plt.text(index, row['so_luong_bill'], f"{int(row['so_luong_bill']):,}", 
    color = 'black', ha = 'center', va = 'bottom')
plt.tight_layout()
plt.savefig(os.path.join('output_plot', 'rank_to_1A.png'), bbox_inches = 'tight') 
plt.show()

In [16]:
a = pd.read_csv('output_all_traces/origin_to_1A_filter.csv')


In [14]:
a = a.sort_values('actual_weight', ascending=False)
print(a[['bill_code', 'actual_weight', 'time_1a_out']][:10])

        bill_code  actual_weight          time_1a_out
545550  B_2212387       19450.00  2026-04-22 17:27:02
454681  B_1842537       18234.00  2026-04-25 02:06:12
477198  B_1937849       18234.00  2026-04-23 19:14:03
477197  B_1937848       18234.00  2026-04-23 19:28:34
434495  B_1751726       18234.00  2026-04-28 19:10:27
477196  B_1937844       18234.00  2026-04-22 20:02:38
489787  B_1983837       18234.00  2026-05-05 15:21:28
545551  B_2212388       18234.00  2026-04-24 19:01:09
477195  B_1937841       18234.00  2026-04-22 15:03:48
412107  B_1670768       15177.72  2026-04-09 21:05:08


In [ ]:
df = a.copy() 
df['time_1a_out'] = pd.to_datetime(df['time_1a_out']) 
df['out_15min'] = df['time_1a_out'].dt.floor('15min') 
trip_df = df.groupby(['kho_o1a', 'out_15min']).agg(
    total_bill = ('bill_code', 'nunique'),
    total_weight = ('actual_weight', 'sum'), 
    avg_wait_time = ('time_in_1a', 'mean')
).reset_index() 

trip_df['avg_wait_time'] = trip_df['avg_wait_time'].round(2)
trip_df['total_weight'] = trip_df['total_weight'].round(2)
trip_df['hour_out'] = trip_df['out_15min'].dt.hour + trip_df['out_15min'].dt.minute / 60.0 
trip_df = trip_df.sort_values(['total_weight', 'total_bill'], ascending=[False,False])
print(trip_df.shape)
trip_df.to_csv(os.path.join("output_all_traces", 'trip_1A.csv'), index = False) 

(18895, 6)
